# Local Training — Surface Crack Detection

**Auto-detect hardware (CUDA / Intel XPU / Apple MPS / CPU) and train.**

| Section | What | Est. Time |
|---------|------|-----------|
| 1 | Device detection + override | — |
| 2 | Dataset check (local or HF download) | ~2 min |
| 3 | Quick test mode (1 epoch) or full run | toggle |
| 4-6 | Train + Evaluate per model | ~15 min full / ~2 min quick |
| 7 | Ensemble | ~30 sec |
| 8 | Summary + Push to HF (optional) | ~30 sec |

In [ ]:
# Cell 1: Add project root to path
import sys, os

PROJECT_DIR = os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f"Project root: {PROJECT_DIR}")
print(f"sys.path OK: {PROJECT_DIR in sys.path}")

In [ ]:
# Cell 2: Auto-detect best device, allow override
import torch
import src.config as cfg

devices = {}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    devices["cuda"] = f"GPU {torch.cuda.get_device_name(0)} ({props.total_memory / 1024**3:.1f} GB)"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    devices["mps"] = "Apple MPS (Metal)"
if hasattr(torch, "xpu") and torch.xpu.is_available():
    devices["xpu"] = "Intel XPU"
devices["cpu"] = "CPU"

keys = list(devices.keys())
print("Detected devices:")
for i, k in enumerate(keys):
    print(f"  [{i}] {k.upper()}: {devices[k]}")

non_cpu = [k for k in keys if k != "cpu"]
default_idx = 0 if non_cpu else keys.index("cpu")
choice = input(f"Select device [0-{len(keys)-1}], Enter=auto ({keys[default_idx].upper()}): ").strip()
if choice:
    selected = keys[int(choice)]
else:
    selected = keys[default_idx]

cfg.Config.DEVICE = torch.device(selected)
print(f"\nUsing: {selected.upper()} — {devices[selected]}")
print(f"Config.DEVICE = {cfg.Config.DEVICE}")

In [ ]:
# Cell 3: Quick test mode toggle
print("Quick test: trains 1 epoch per model (checks everything works).")
print("Full run: 5 warmup + 15 fine-tune epochs per model.")
choice = input("Run quick test? (y/N): ").strip().lower()

QUICK_MODE = choice in ("y", "yes")
if QUICK_MODE:
    cfg.Config.EPOCHS_WARMUP = 1
    cfg.Config.EPOCHS_FINE_TUNE = 1
    cfg.Config.EARLY_STOP_PATIENCE = 5  # prevent early stop in quick mode
    print("\nQuick mode: 1 warmup + 1 fine-tune epoch.")
else:
    print("\nFull mode: 5 warmup + 15 fine-tune epochs.")

print(f"  Warmup: {cfg.Config.EPOCHS_WARMUP}, Fine-tune: {cfg.Config.EPOCHS_FINE_TUNE}")

In [ ]:
# Cell 4: Check local dataset, download from HF fallback
import zipfile
from huggingface_hub import hf_hub_download

RAW_DIR = "data/raw"
REPO_ID = "amruthjakku/surface-crack-detection-model"

if os.path.exists(RAW_DIR) and os.listdir(RAW_DIR):
    n = len(os.listdir(RAW_DIR))
    print(f"Dataset found locally: {RAW_DIR}/ ({n} class folders)")
else:
    print(f"Local dataset not found at {RAW_DIR}/")
    dl = input("Download from HF Hub? (Y/n): ").strip().lower()
    if dl not in ("n", "no"):
        try:
            print("Downloading...")
            os.makedirs("data", exist_ok=True)
            zip_path = hf_hub_download(repo_id=REPO_ID, filename="dataset.zip", repo_type="model")
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall("data")
            print(f"Extracted to {RAW_DIR}/")
        except Exception as e:
            print(f"Download failed: {e}")
            print("Place images in data/raw/ and re-run.")
            raise SystemExit(1)
    else:
        print("Place images in data/raw/ with subfolders:")
        print("  data/raw/Cracks/, data/raw/Patch/, data/raw/Potholes/, data/raw/Surface Defects/")
        raise SystemExit(1)

from src.prepare_data import prepare_data
prepare_data()
print("Data prepared.")

from src.dataset import get_dataloaders
train_loader, val_loader, test_loader = get_dataloaders()
print(f"Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")
if len(train_loader.dataset) == 0:
    print("ERROR: Training set empty. Check data/raw/ contents.")
    raise SystemExit(1)

In [ ]:
# Cell 5: Train + Evaluate ResNet50
from src.train import run_training
from src.evaluate import run_evaluation
import time as _time

print("=" * 60)
print("ResNet50")
print("=" * 60)
t0 = _time.time()
try:
    run_training(model_name="resnet50")
    elapsed = _time.time() - t0
    print(f"\nTraining time: {elapsed / 60:.1f} min")
    # Evaluate
    eval_path = cfg.Config.get_model_path("resnet50")
    if os.path.exists(eval_path):
        print(f"Checkpoint: {eval_path} ({os.path.getsize(eval_path)/1024/1024:.1f} MB)")
        run_evaluation(model_name="resnet50")
    else:
        print("No checkpoint saved.")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"ResNet50 failed: {e}")
torch.cuda.empty_cache()

In [ ]:
# Cell 6: Train + Evaluate EfficientNet-B0
print("=" * 60)
print("EfficientNet-B0")
print("=" * 60)
t0 = _time.time()
try:
    run_training(model_name="efficientnet_b0")
    elapsed = _time.time() - t0
    print(f"\nTraining time: {elapsed / 60:.1f} min")
    eval_path = cfg.Config.get_model_path("efficientnet_b0")
    if os.path.exists(eval_path):
        print(f"Checkpoint: {eval_path} ({os.path.getsize(eval_path)/1024/1024:.1f} MB)")
        run_evaluation(model_name="efficientnet_b0")
    else:
        print("No checkpoint saved.")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"EfficientNet-B0 failed: {e}")
torch.cuda.empty_cache()

In [ ]:
# Cell 7: Train + Evaluate ViT-B/16
print("=" * 60)
print("ViT-B/16")
print("=" * 60)
t0 = _time.time()
try:
    run_training(model_name="vit_b_16")
    elapsed = _time.time() - t0
    print(f"\nTraining time: {elapsed / 60:.1f} min")
    eval_path = cfg.Config.get_model_path("vit_b_16")
    if os.path.exists(eval_path):
        print(f"Checkpoint: {eval_path} ({os.path.getsize(eval_path)/1024/1024:.1f} MB)")
        run_evaluation(model_name="vit_b_16")
    else:
        print("No checkpoint saved.")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"ViT-B/16 failed: {e}")
torch.cuda.empty_cache()

In [ ]:
# Cell 8: Ensemble evaluation
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from src.dataset import get_dataloaders
from src.model import get_model

models_to_load = [m for m in cfg.Config.ENSEMBLE_MODELS if os.path.exists(cfg.Config.get_model_path(m))]

if len(models_to_load) < 2:
    print(f"Ensemble skipped — need >= 2 models, have {len(models_to_load)}")
else:
    device = cfg.Config.DEVICE
    nets = []
    for name in models_to_load:
        net = get_model(model_name=name, num_classes=cfg.Config.NUM_CLASSES, pretrained=False)
        net.load_state_dict(torch.load(cfg.Config.get_model_path(name), map_location=device))
        net = net.to(device).eval()
        nets.append(net)

    _, _, test_loader = get_dataloaders()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            probs = torch.zeros(images.size(0), cfg.Config.NUM_CLASSES).to(device)
            for net in nets:
                probs += torch.softmax(net(images), dim=1)
            probs /= len(nets)
            y_true.extend(labels.numpy())
            y_pred.extend(torch.max(probs, 1)[1].cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    print(f"\n{'=' * 60}")
    print(f"Ensemble ({'+'.join(models_to_load)}) Results")
    print(f"{'=' * 60}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=cfg.Config.CLASSES)}")

In [ ]:
# Cell 9: Summary
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Device: {cfg.Config.DEVICE}")
print(f"Mode: {'Quick test (1 epoch)' if QUICK_MODE else 'Full training'}")
print()
for name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
    path = cfg.Config.get_model_path(name)
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1024 / 1024
        print(f"  {name:20s} {mb:.1f} MB")
    else:
        print(f"  {name:20s} not trained")
if QUICK_MODE:
    print("\nQuick test complete. Run again with quick=no for full training.")

In [ ]:
# Cell 10: Push to HF Hub (optional)
token = input("Enter HF token (write access) or Enter to skip: ").strip()
if token:
    from huggingface_hub import HfApi, login
    try:
        login(token=token)
        api = HfApi()
        for name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
            path = cfg.Config.get_model_path(name)
            if not os.path.exists(path):
                continue
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=f"{name}_best.pth",
                repo_id="amruthjakku/surface-crack-detection-model",
                repo_type="model",
            )
            print(f"Pushed {name}_best.pth")
        print("Done.")
    except Exception as e:
        print(f"Push failed: {e}")
else:
    print("Skipped.")

In [ ]:
# Cell 11: Done
print("=" * 60)
print("Local training complete!")
print("=" * 60)
print("Models saved in models/")
print("Reports saved in reports/")
print("\nTo compare results across runs, run notebooks/02_results.ipynb")